# 🛩️ Drone Emergency Landing Zone Prediction
## Full Pipeline Demo Notebook
### Dataset: Aerial Semantic Segmentation Drone Dataset (Kaggle)

This notebook walks through the complete pipeline:
1. Dataset loading and EDA
2. Segmentation model training (Module 1)
3. Weather model training (Module 2)
4. Threat model training (Module 3)
5. End-to-end inference + visualization

In [ ]:
# ── Install dependencies (uncomment on Kaggle/Colab) ──
# !pip install keras-segmentation albumentations segmentation-models-pytorch

import os, sys, json, yaml, time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
import tensorflow as tf

# If running from notebooks/ dir, add parent to path
sys.path.insert(0, str(Path.cwd().parent))

print(f'TF version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 1. Dataset Exploration

In [ ]:
# ── Paths — adjust to your Kaggle dataset location ──
# On Kaggle: /kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/
# Locally: data/raw/semantic_drone_dataset/

DATASET_ROOT = "/kaggle/input/semantic-drone-dataset/dataset/semantic_drone_dataset/"
# DATASET_ROOT = "../data/raw/semantic_drone_dataset/"  # Local

orig_dir  = Path(DATASET_ROOT) / "original_images"
label_dir = Path(DATASET_ROOT) / "label_images_semantic"
class_csv = Path(DATASET_ROOT) / "class_dict_seg.csv"

images = sorted(orig_dir.glob("*.jpg"))
masks  = sorted(label_dir.glob("*.png"))

print(f"Images: {len(images)}")
print(f"Masks:  {len(masks)}")

# Load class dictionary
class_df = pd.read_csv(class_csv)
print(f"\nClasses ({len(class_df)}):")
class_df

In [ ]:
# ── Visualize sample image + mask ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sample_img  = np.array(Image.open(images[0]))
sample_mask = np.array(Image.open(masks[0]))

axes[0].imshow(sample_img)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(sample_mask)
axes[1].set_title('Semantic Mask (RGB)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(sample_img)
axes[2].imshow(sample_mask, alpha=0.6)
axes[2].set_title('Overlay', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 2. Build & Train Segmentation Model (Module 1)

In [ ]:
# Using keras-segmentation library (matches the Kaggle notebook)
!pip install keras-segmentation -q

from keras_segmentation.models.unet import resnet50_unet

N_CLASSES  = 24
IMG_HEIGHT = 416
IMG_WIDTH  = 608

model = resnet50_unet(n_classes=N_CLASSES, input_height=IMG_HEIGHT, input_width=IMG_WIDTH)
print('Model built successfully')

In [ ]:
# ── Train segmentation model ──
# NOTE: On Kaggle, this runs on GPU P100 (takes ~2hrs for 10 epochs)

TRAIN_IMAGES = str(orig_dir) + '/'
TRAIN_MASKS  = str(label_dir) + '/'
CHECKPOINT   = 'resnet50_unet'
EPOCHS       = 10

model.train(
    train_images=TRAIN_IMAGES,
    train_annotations=TRAIN_MASKS,
    checkpoints_path=CHECKPOINT,
    epochs=EPOCHS
)
print('Training complete!')

In [ ]:
# ── Predict on sample image ──
import time

t0  = time.time()
out = model.predict_segmentation(
    inp=str(images[0]),
    out_fname="/tmp/seg_output.png"
)
print(f'Inference time: {time.time()-t0:.2f}s')
print(f'Output shape: {out.shape}')
print(f'Unique classes predicted: {np.unique(out)}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
img_orig = np.array(Image.open(images[0]))
axes[0].imshow(img_orig)
axes[0].imshow(out, alpha=0.6)
axes[0].set_title('Prediction Overlay')
axes[0].axis('off')

img_gt = np.array(Image.open(masks[0]))
axes[1].imshow(img_orig)
axes[1].imshow(img_gt, alpha=0.6)
axes[1].set_title('Ground Truth Overlay')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Terrain Safety Analysis

In [ ]:
# ── Class safety mapping ──
CLASS_SAFETY = {
    0: 0.0,  # unlabeled
    1: 1.0,  # paved-area ✅
    2: 0.9,  # dirt ✅
    3: 0.85, # grass ✅
    4: 0.7,  # gravel ⚠️
    5: 0.0,  # water ❌
    6: 0.1,  # rocks ❌
    7: 0.0,  # pool ❌
    8: 0.6,  # vegetation ⚠️
    9: 0.5,  # roof ⚠️
    10: 0.0, 11: 0.0, 12: 0.0, 13: 0.0, 14: 0.0,
    15: 0.0, 16: 0.0, 17: 0.0, 18: 0.0, 19: 0.4,
    20: 0.0, 21: 0.0, 22: 0.0, 23: 0.0
}

def build_safety_map(class_mask):
    h, w = class_mask.shape
    safety = np.zeros((h, w), dtype=np.float32)
    for cls, val in CLASS_SAFETY.items():
        safety[class_mask == cls] = val
    return safety

safety_map = build_safety_map(out)

# Visualize safety heatmap
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('safety', [
    (0, '#FF2200'), (0.4, '#FF8800'), (0.6, '#FFEE00'), (1, '#00CC44')
])

plt.figure(figsize=(12, 7))
plt.imshow(safety_map, cmap=cmap, vmin=0, vmax=1)
plt.colorbar(label='Safety Score (0=Danger, 1=Safe)')
plt.title('Terrain Safety Map', fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

print(f'Mean terrain safety: {safety_map.mean():.3f}')
print(f'Max terrain safety:  {safety_map.max():.3f}')

In [ ]:
# ── Find candidate landing zones using connected components ──
from scipy import ndimage

# Smooth and threshold
smooth  = ndimage.gaussian_filter(safety_map, sigma=10)
binary  = (smooth >= 0.60).astype(np.uint8)

# Connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary)

candidates = []
for lbl in range(1, num_labels):
    area = stats[lbl, cv2.CC_STAT_AREA]
    if area < 500:
        continue
    cx, cy = centroids[lbl]
    region_scores = safety_map[labels == lbl]
    candidates.append({
        'centroid': (int(cx), int(cy)),
        'score': float(np.mean(region_scores)),
        'area': int(area)
    })

candidates.sort(key=lambda z: z['score'], reverse=True)
print(f'Found {len(candidates)} candidate zones')
for i, c in enumerate(candidates[:5]):
    print(f'  Zone {i+1}: centroid={c["centroid"]}, score={c["score"]:.3f}, area={c["area"]}px')

## 4. Weather + Threat Scoring (Modules 2 & 3)

In [ ]:
# ── Weather safety (rule-based) ──
def weather_safety_rule_based(weather):
    vis    = weather.get('visibility', 10.0)
    wind   = weather.get('wind_speed', 0.0)
    precip = weather.get('precipitation', 0.0)
    fog    = weather.get('fog_index', 0.0)
    smoke  = weather.get('smoke_density', 0.0)
    risk = (
        max(0, 1 - vis/10.0) * 0.30 +
        min(1, wind/15.0)    * 0.25 +
        min(1, precip/20.0)  * 0.20 +
        fog                  * 0.15 +
        smoke                * 0.10
    )
    return float(1.0 - np.clip(risk, 0, 1))

# ── Threat safety (rule-based) ──
def threat_safety_rule_based(threat):
    score = (
        threat.get('hostile_area', 0) * 0.35 +
        threat.get('prohibited_zone', 0) * 0.20 +
        threat.get('mine_region', 0) * 0.20 +
        threat.get('gunfire_probability', 0) * 0.15 +
        threat.get('blast_radius', 0) * 0.10
    )
    return float(1.0 - np.clip(score, 0, 1))

# Sample conditions
sample_weather = {
    'visibility': 4.5, 'wind_speed': 8.2, 'precipitation': 2.1,
    'fog_index': 0.3, 'smoke_density': 0.1
}
sample_threat = {
    'hostile_area': 0.3, 'prohibited_zone': 0.1,
    'mine_region': 0.05, 'gunfire_probability': 0.2, 'blast_radius': 0.15
}

weather_s = weather_safety_rule_based(sample_weather)
threat_s  = threat_safety_rule_based(sample_threat)

print(f'Weather safety score: {weather_s:.3f}')
print(f'Threat safety score:  {threat_s:.3f}')

## 5. Decision Engine — Final Ranking

In [ ]:
# ── Combined scoring and ranking ──
W_TERRAIN = 0.50
W_WEATHER = 0.30
W_THREAT  = 0.20

ranked_zones = []
for i, zone in enumerate(candidates[:10]):
    cx, cy = zone['centroid']
    terrain_s = float(safety_map[cy, cx]) if cy < safety_map.shape[0] and cx < safety_map.shape[1] else zone['score']
    final = W_TERRAIN * terrain_s + W_WEATHER * weather_s + W_THREAT * threat_s
    ranked_zones.append({
        'rank': i+1,
        'centroid': zone['centroid'],
        'terrain_score': round(terrain_s, 3),
        'weather_score': round(weather_s, 3),
        'threat_score':  round(threat_s, 3),
        'final_score':   round(final, 3),
        'area': zone['area']
    })

ranked_zones.sort(key=lambda z: z['final_score'], reverse=True)

print('TOP 3 LANDING ZONES')
print('=' * 60)
for zone in ranked_zones[:3]:
    print(f"  Rank #{zone['rank']}: {zone['centroid']}  |  "
          f"Terrain={zone['terrain_score']}  Weather={zone['weather_score']}  "
          f"Threat={zone['threat_score']}  FINAL={zone['final_score']}")

In [ ]:
# ── Final annotated visualization ──
img_resized = cv2.resize(np.array(Image.open(images[0])), (out.shape[1], out.shape[0]))

# Build combined heatmap
threat_grid    = np.full_like(safety_map, threat_s)
weather_map    = np.full_like(safety_map, weather_s)
combined_heatmap = (
    W_TERRAIN * safety_map +
    W_WEATHER * weather_map +
    W_THREAT  * threat_grid
)

from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('safety', [
    (0, '#FF2200'), (0.4, '#FF8800'), (0.6, '#FFEE00'), (1, '#00CC44')
])

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.patch.set_facecolor('#0A0E1A')

# Original
axes[0].imshow(img_resized)
axes[0].set_title('Original Image', color='white', fontsize=12)
axes[0].axis('off')

# Combined heatmap
axes[1].imshow(combined_heatmap, cmap=cmap, vmin=0, vmax=1)
axes[1].set_title('Combined Risk Heatmap', color='white', fontsize=12)
axes[1].axis('off')

# Annotated with zones
annotated = img_resized.copy()
colors = [(0, 220, 60), (255, 220, 0), (255, 120, 0)]
for i, zone in enumerate(ranked_zones[:3]):
    cx, cy = zone['centroid']
    color  = colors[min(i, 2)]
    cv2.circle(annotated, (cx, cy), 20, color, -1)
    cv2.circle(annotated, (cx, cy), 22, (255,255,255), 2)
    cv2.putText(annotated, f"#{i+1} {zone['final_score']:.2f}",
                (cx-30, cy-28), cv2.FONT_HERSHEY_DUPLEX, 0.55, color, 1)

axes[2].imshow(annotated)
axes[2].set_title('Ranked Landing Zones', color='white', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('landing_zone_analysis.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Analysis complete!')

## 6. Train ML-based Weather + Threat Models

In [ ]:
# Generate synthetic training data and train MLP models
import sys
sys.path.insert(0, '..')

# Uncomment if running standalone (not from project root)
# from data.prepare_dataset import generate_weather_dataset, generate_threat_dataset
# generate_weather_dataset()
# generate_threat_dataset()

# Then:
# from models.weather_model import train_weather_model
# from models.threat_model import train_threat_model
# wx_model, wx_hist = train_weather_model()
# th_model, th_hist = train_threat_model()

print('All modules ready!')
print('Run train.py for full training pipeline.')